# Kickstarter EDA — Distribution & Relationship Plots
Dataset: `kickstarter_raw_before_encoding.csv` (raw, pre-encoding, 20,000 campaigns)

Feature choices guided by SHAP importance (top interpretable, raw-available drivers of `log_target`):
`log_goal_usd`, `has_video`, `category_parent`, `launch_year`, `prelaunch_days`, `duration_days`.

Paired plots of the same type are now combined side-by-side into a single figure (no figure-number titles; only axis labels and legends).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "font.family": "DejaVu Sans",
})

PALETTE = "viridis"
FIGSIZE = (8, 5.5)

df = pd.read_csv(r"E:\NSU\cse445\EDA attempt3\Dataset\kickstarter_raw_before_encoding.csv")
df = df.dropna(subset=["log_goal_usd", "log_target"]).copy()

df["log_prelaunch_days"] = np.log1p(df["prelaunch_days"])

top_categories = df["category_parent"].value_counts().nlargest(8).index
df["category_parent_top"] = np.where(df["category_parent"].isin(top_categories), df["category_parent"], "Other")

df.shape

## Scatter Plot — Goal vs Funding (No Hue / Hue = Category)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

sns.scatterplot(data=df, x="log_goal_usd", y="log_target", alpha=0.35, s=20, color="#3b6fa0", ax=axes[0])
axes[0].set_xlabel("log(Goal USD)")
axes[0].set_ylabel("log(Target/Funded USD)")

sns.scatterplot(data=df, x="log_goal_usd", y="log_target", hue="category_parent_top",
                 palette=PALETTE, alpha=0.45, s=20, ax=axes[1])
axes[1].set_xlabel("log(Goal USD)")
axes[1].set_ylabel("log(Target/Funded USD)")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Category", frameon=False)

plt.tight_layout()
plt.savefig("fig1_scatter_goal_vs_target_combined.png", bbox_inches="tight")
plt.show()

## KDE Plot — Funding Distribution (No Hue / Hue = Has Video)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

sns.kdeplot(data=df, x="log_target", fill=True, color="#3b6fa0", linewidth=2, ax=axes[0])
axes[0].set_xlabel("log(Target/Funded USD)")

sns.kdeplot(data=df, x="log_target", hue="has_video", fill=True,
            palette={0: "#d95f5f", 1: "#3ba272"}, alpha=0.4, linewidth=2, ax=axes[1], legend=False)
axes[1].set_xlabel("log(Target/Funded USD)")
handles = [plt.Line2D([0], [0], color=c, lw=4) for c in ["#d95f5f", "#3ba272"]]
axes[1].legend(handles, ["No Video", "Has Video"], title="Video", frameon=False)

plt.tight_layout()
plt.savefig("fig2_kde_target_combined.png", bbox_inches="tight")
plt.show()

## Histogram — Goal Distribution (No KDE / With KDE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

sns.histplot(df["log_goal_usd"], bins=40, color="#3b6fa0", kde=False, ax=axes[0])
axes[0].set_xlabel("log(Goal USD)")

sns.histplot(df["log_goal_usd"], bins=40, color="#3b6fa0", kde=True,
             line_kws={"linewidth": 2, "color": "#1a1a1a"}, ax=axes[1])
axes[1].set_xlabel("log(Goal USD)")

plt.tight_layout()
plt.savefig("fig3_hist_goal_combined.png", bbox_inches="tight")
plt.show()

## Histogram — Prelaunch Days Distribution (No KDE / With KDE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

sns.histplot(df["log_prelaunch_days"], bins=40, color="#e0954f", kde=False, ax=axes[0])
axes[0].set_xlabel("log(1 + Prelaunch Days)")

sns.histplot(df["log_prelaunch_days"], bins=40, color="#e0954f", kde=True,
             line_kws={"linewidth": 2, "color": "#1a1a1a"}, ax=axes[1])
axes[1].set_xlabel("log(1 + Prelaunch Days)")

plt.tight_layout()
plt.savefig("fig4_hist_prelaunch_combined.png", bbox_inches="tight")
plt.show()

## Boxplot — Funding by Category / Funding by Launch Year

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(19, 6))

order_cat = df.groupby("category_parent_top")["log_target"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="category_parent_top", y="log_target", order=order_cat, palette=PALETTE, fliersize=2, ax=axes[0])
axes[0].set_xlabel("Category")
axes[0].set_ylabel("log(Target/Funded USD)")
axes[0].tick_params(axis="x", rotation=30)
for label in axes[0].get_xticklabels():
    label.set_ha("right")

recent = df[df["launch_year"].between(2014, 2025)]
sns.boxplot(data=recent, x="launch_year", y="log_target", palette="crest", fliersize=2, ax=axes[1])
axes[1].set_xlabel("Launch Year")
axes[1].set_ylabel("log(Target/Funded USD)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("fig5_boxplot_category_year_combined.png", bbox_inches="tight")
plt.show()

## Scatter Plot — Duration vs Funding (No Hue / Hue = Prelaunch Activated)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

sns.scatterplot(data=df, x="duration_days", y="log_target", alpha=0.35, s=20, color="#3b6fa0", ax=axes[0])
axes[0].set_xlabel("Campaign Duration (days)")
axes[0].set_ylabel("log(Target/Funded USD)")

sns.scatterplot(data=df, x="duration_days", y="log_target", hue="prelaunch_activated",
                 palette={0: "#d95f5f", 1: "#3ba272"}, alpha=0.45, s=20, ax=axes[1], legend=False)
axes[1].set_xlabel("Campaign Duration (days)")
axes[1].set_ylabel("log(Target/Funded USD)")
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8) for c in ["#d95f5f", "#3ba272"]]
axes[1].legend(handles, ["No", "Yes"], title="Prelaunch\nActivated", frameon=False)

plt.tight_layout()
plt.savefig("fig6_scatter_duration_combined.png", bbox_inches="tight")
plt.show()